# Model Experiments

This notebook is used to experiment with and optimize the selected models:

- Random Forest
- Ridge Regression
- TensorFlow Dense Neural Network

Each model will be trained and improved independently, with a focus on generalization and validation performance. Models will be compared against previous configurations throughout the experimentation process.

The final comparison will be performed once further improvements become marginal relative to the effort required.

In [26]:
from keras import Sequential, Input
from keras.src.callbacks import EarlyStopping
from keras.src.layers import Dense, Dropout
from keras.src.optimizers import Adam
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from tensorflow.python.keras.regularizers import l2

from src.common.constants import FINAL_SELECTED_FEATURES
from src.common.schemas import DeepLearningFitParamSchema
from src.modeling.data_utils import load_modeling_data, split_modeling_data, train_and_evaluate_model, preprocess_data

In [27]:
df = load_modeling_data()

train_df, val_df, test_df = split_modeling_data(df)

## Experiment 1 — Random Forest

Train and optimize the Random Forest model as a strong baseline.

Focus on:
- Hyperparameter tuning
- Validation performance
- Generalization
- Avoiding overfitting

Iterate on the configuration until further improvements become marginal.

In [28]:
rf_base_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features=1.0,
    max_samples=1.0,
    random_state=42,
    n_jobs=-1,
)

rf_base_model, (rf_base_rmse, rf_base_mae, rf_base_r2) = train_and_evaluate_model(train_df, val_df,
                         label="Random Forest Base Model", toggle_evaluate_print=True, disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=rf_base_model)


target_aqi_day1
  RMSE: 16.0476
  MAE:  10.3983
  R²:   0.8372

target_aqi_day2
  RMSE: 28.2155
  MAE:  19.3990
  R²:   0.5048

target_aqi_day3
  RMSE: 31.3263
  MAE:  22.5330
  R²:   0.4227

target_aqi_day4
  RMSE: 31.7258
  MAE:  23.5333
  R²:   0.4320


**Observation:** Random Forest performs reasonably well for the 1-day-ahead target (`R² = 0.8372`, `RMSE = 16.0476`, `MAE = 10.3983`) but performance drops substantially for Days 2–4, with `R²` falling to `0.42–0.50` and errors increasing. This suggests the current configuration captures short-term AQI patterns better than longer-horizon patterns, so further Random Forest hyperparameter tuning is required.

In [29]:
rf_base_model_r = RandomForestRegressor(
    n_estimators=600,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    max_samples=0.9,
    random_state=42,
    n_jobs=-1,
)

rf_base_model_r, (rf_base_rmse_r, rf_base_mae_r, rf_base_r2_r) = train_and_evaluate_model(train_df, val_df,
                         label="Random Forest Base Model Regularized", disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=rf_base_model_r,
                         compare_model=("Random Forest Base Model", (rf_base_rmse, rf_base_mae, rf_base_r2)))

,Target,Random Forest Base Model RMSE,Random Forest Base Model MAE,Random Forest Base Model R²,Random Forest Base Model Regularized RMSE,Random Forest Base Model Regularized MAE,Random Forest Base Model Regularized R²
0,target_aqi_day1,16.047587,10.398290,0.837215,15.816504,10.285093,0.841869
1,target_aqi_day2,28.215522,19.399020,0.504783,27.641673,19.082047,0.524722
2,target_aqi_day3,31.326312,22.533011,0.422727,31.105018,22.321087,0.430854
3,target_aqi_day4,31.725815,23.533278,0.431963,31.866488,23.707509,0.426915
4,Mean,26.828809,18.965900,0.549172,26.607421,18.848934,0.556090


**Observation:** Regularization improved the Random Forest slightly, increasing mean R² from `0.5492` to `0.5561` and reducing mean RMSE from `26.8288` to `26.6074`. The improvement is consistent for Days 1–3 but negligible for Day 4, indicating that the regularization approach is beneficial but the current parameter changes have only a limited impact. Stronger regularization will therefore be tested next.

In [30]:
rf_base_model_r_2 = RandomForestRegressor(
    n_estimators=600,
    max_depth=20,
    min_samples_split=7,
    min_samples_leaf=7,
    max_features=0.85,
    max_samples=0.9,
    random_state=42,
    n_jobs=-1,
)

rf_base_model_r_2, (rf_base_rmse_r_2, rf_base_mae_r_2, rf_base_r2_r_2) = train_and_evaluate_model(train_df, val_df,
                         label="Random Forest Base Model Regularized v2", disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=rf_base_model_r_2,
                         compare_model=("Random Forest Base Model Regularized", (rf_base_rmse_r, rf_base_mae_r, rf_base_r2_r)))

,Target,Random Forest Base Model Regularized RMSE,Random Forest Base Model Regularized MAE,Random Forest Base Model Regularized R²,Random Forest Base Model Regularized v2 RMSE,Random Forest Base Model Regularized v2 MAE,Random Forest Base Model Regularized v2 R²
0,target_aqi_day1,15.816504,10.285093,0.841869,15.767436,10.236939,0.842849
1,target_aqi_day2,27.641673,19.082047,0.524722,27.292848,18.974586,0.536642
2,target_aqi_day3,31.105018,22.321087,0.430854,30.923929,22.263238,0.437462
3,target_aqi_day4,31.866488,23.707509,0.426915,31.818725,23.738206,0.428632
4,Mean,26.607421,18.848934,0.556090,26.450734,18.803242,0.561396


**Observation:** The second regularization experiment produced only a small improvement over the previous configuration, with mean RMSE decreasing from `26.6074` to `26.4507`, MAE from `18.8489` to `18.8032`, and R² increasing from `0.5561` to `0.5614`. Although the changes to `min_samples_split`, `min_samples_leaf`, `max_features`, and `max_samples` improved the overall result, their individual impact remains limited. The current configuration will be used as the benchmark for the final `n_estimators` experiment.

In [31]:
rf_base_model_r_3 = RandomForestRegressor(
    n_estimators=800,
    max_depth=20,
    min_samples_split=7,
    min_samples_leaf=7,
    max_features=0.85,
    max_samples=0.9,
    random_state=42,
    n_jobs=-1,
)

rf_base_model_r_3, (rf_base_rmse_r_3, rf_base_mae_r_3, rf_base_r2_r_3) = train_and_evaluate_model(train_df, val_df,
                         label="Random Forest Base Model Regularized v3", disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=rf_base_model_r_3,
                         compare_model=("Random Forest Base Model Regularized v2", (
                                        rf_base_rmse_r_2, rf_base_mae_r_2, rf_base_r2_r_2
                         )))

,Target,Random Forest Base Model Regularized v2 RMSE,Random Forest Base Model Regularized v2 MAE,Random Forest Base Model Regularized v2 R²,Random Forest Base Model Regularized v3 RMSE,Random Forest Base Model Regularized v3 MAE,Random Forest Base Model Regularized v3 R²
0,target_aqi_day1,15.767436,10.236939,0.842849,15.773150,10.232304,0.842735
1,target_aqi_day2,27.292848,18.974586,0.536642,27.296457,19.018086,0.536519
2,target_aqi_day3,30.923929,22.263238,0.437462,30.918054,22.297219,0.437675
3,target_aqi_day4,31.818725,23.738206,0.428632,31.794940,23.745205,0.429486
4,Mean,26.450734,18.803242,0.561396,26.445650,18.823203,0.561604


**Observation:** Increasing `n_estimators` from `600` to `800` produced negligible changes in performance, with mean RMSE improving from `26.4507` to `26.4457` and mean R² from `0.5614` to `0.5616`, while mean MAE slightly worsened. This indicates that increasing the number of trees has reached diminishing returns, so further Random Forest tuning is not justified.

## Experiment 2 — Ridge Regression

Train and optimize the Ridge Regression model as a linear baseline.

Focus on:
- Preprocessing and scaling
- Regularization strength
- Validation performance
- Generalization
- Avoiding overfitting

Iterate on the configuration until further improvements become marginal.

In [32]:
train_df, val_df, test_df = preprocess_data(train_df, val_df, test_df)

In [33]:
ridge_base_model = Ridge(
    alpha=10.0,
    fit_intercept=True,
    solver="auto",
    tol=1e-4,
    max_iter=5000,
)

ridge_base_model, (ridge_base_rmse, ridge_base_mae, ridge_base_r2) = train_and_evaluate_model(train_df, val_df,
                         label="Ridge Regression Base Model", disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=ridge_base_model,
                         compare_model=("Random Forest Base Model Regularized v3", (
                                        rf_base_rmse_r_3, rf_base_mae_r_3, rf_base_r2_r_3
                         )))

,Target,Random Forest Base Model Regularized v3 RMSE,Random Forest Base Model Regularized v3 MAE,Random Forest Base Model Regularized v3 R²,Ridge Regression Base Model RMSE,Ridge Regression Base Model MAE,Ridge Regression Base Model R²
0,target_aqi_day1,15.773150,10.232304,0.842735,14.398813,10.631519,0.868947
1,target_aqi_day2,27.296457,19.018086,0.536519,26.640181,18.573379,0.558538
2,target_aqi_day3,30.918054,22.297219,0.437675,30.287032,21.520296,0.460395
3,target_aqi_day4,31.794940,23.745205,0.429486,31.786017,23.061679,0.429806
4,Mean,26.445650,18.823203,0.561604,25.778011,18.446718,0.579421


**Observation:** Ridge Regression outperformed the final Random Forest configuration across the mean metrics, improving RMSE from `26.4457` to `25.7780`, MAE from `18.8232` to `18.4467`, and R² from `0.5616` to `0.5794`. The largest improvement was observed for the 1-day-ahead target, while Day 4 remained nearly unchanged. Further Ridge optimization will focus on regularization strength (`alpha`).

In [34]:
ridge_base_model_r = Ridge(
    alpha=0.01,
    fit_intercept=True,
    solver="auto",
    tol=1e-4,
    max_iter=5000,
)

ridge_base_model_r, (ridge_base_rmse_r, ridge_base_mae_r, ridge_base_r2_r) = train_and_evaluate_model(train_df, val_df,
                         label="Ridge Regression Base Model Regularized", disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=ridge_base_model_r,
                         compare_model=("Ridge Regression Base Model", (
                                        ridge_base_rmse, ridge_base_mae, ridge_base_r2
                         )))

,Target,Ridge Regression Base Model RMSE,Ridge Regression Base Model MAE,Ridge Regression Base Model R²,Ridge Regression Base Model Regularized RMSE,Ridge Regression Base Model Regularized MAE,Ridge Regression Base Model Regularized R²
0,target_aqi_day1,14.398813,10.631519,0.868947,14.265966,10.489315,0.871354
1,target_aqi_day2,26.640181,18.573379,0.558538,26.840461,18.772470,0.551875
2,target_aqi_day3,30.287032,21.520296,0.460395,30.109489,21.517893,0.466703
3,target_aqi_day4,31.786017,23.061679,0.429806,31.580063,22.956517,0.437171
4,Mean,25.778011,18.446718,0.579421,25.698995,18.434049,0.581776


**Observation:** Reducing Ridge's `alpha` to `0.01` produced only a marginal improvement in mean performance, increasing R² from `0.5794` to `0.5818` and reducing RMSE from `25.7780` to `25.6990`. The improvement is too small to justify further `alpha` tuning. The next experiment will test `max_iter` once to verify whether additional iterations affect convergence or performance.

In [35]:
ridge_base_model_r_2 = Ridge(
    alpha=0.01,
    fit_intercept=True,
    solver="auto",
    tol=1e-4,
    max_iter=10000,
)

ridge_base_model_r_2, (ridge_base_rmse_r_2, ridge_base_mae_r_2, ridge_base_r2_r_2) = train_and_evaluate_model(train_df, val_df,
                         label="Ridge Regression Base Model Regularized v2", disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=ridge_base_model_r_2,
                         compare_model=("Ridge Regression Base Model Regularized", (
                                        ridge_base_rmse_r, ridge_base_mae_r, ridge_base_r2_r
                         )))

,Target,Ridge Regression Base Model Regularized RMSE,Ridge Regression Base Model Regularized MAE,Ridge Regression Base Model Regularized R²,Ridge Regression Base Model Regularized v2 RMSE,Ridge Regression Base Model Regularized v2 MAE,Ridge Regression Base Model Regularized v2 R²
0,target_aqi_day1,14.265966,10.489315,0.871354,14.265966,10.489315,0.871354
1,target_aqi_day2,26.840461,18.772470,0.551875,26.840461,18.772470,0.551875
2,target_aqi_day3,30.109489,21.517893,0.466703,30.109489,21.517893,0.466703
3,target_aqi_day4,31.580063,22.956517,0.437171,31.580063,22.956517,0.437171
4,Mean,25.698995,18.434049,0.581776,25.698995,18.434049,0.581776


**Observation:** Increasing Ridge's `max_iter` produced no change in performance, confirming that the model is already converging before reaching the iteration limit. Further Ridge optimization is therefore unlikely to provide meaningful gains through these hyperparameters.

## Experiment 3 — Dense Neural Network (MLP)

Train and optimize the MLP model as the deep learning model.

Focus on:
- Architecture and capacity
- Regularization
- Learning rate and training configuration
- Validation performance
- Generalization
- Avoiding overfitting

Iterate on the configuration until further improvements become marginal.

In [36]:
mlp_base_model = Sequential([
    Input(shape=(23,)),
    Dense(
        8,
        activation="relu",
        kernel_regularizer=l2(1e-4),
    ),
    Dropout(0.15),
    Dense(4, activation="linear"),
])

mlp_base_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="mse",
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
)

mlp_base_model_fit_params: DeepLearningFitParamSchema = {
    "epochs": 500,
    "batch_size": 32,
    "callbacks": [early_stopping],
    "verbose": 1
}

mlp_base_model, (mlp_base_rmse, mlp_base_mae, mlp_base_r2) = train_and_evaluate_model(train_df, val_df,
                         label="MLP Base Model", disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=mlp_base_model,
                         compare_model=("Ridge Regression Base Model Regularized", (
                                        ridge_base_rmse_r, ridge_base_mae_r, ridge_base_r2_r
                         )),
                         deep_learning=mlp_base_model_fit_params)

Epoch 1/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 25976.8770 - val_loss: 25855.6230
Epoch 2/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25886.0137 - val_loss: 25790.6250
Epoch 3/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25809.0625 - val_loss: 25722.8027
Epoch 4/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25720.5391 - val_loss: 25644.6758
Epoch 5/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25620.4160 - val_loss: 25550.8379
Epoch 6/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25515.8535 - val_loss: 25440.0293
Epoch 7/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25373.8555 - val_loss: 25306.7031
Epoch 8/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25219.6758 - val_loss: 25149.5215
Epoch 9/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25035.5938 - val_loss: 24962.2656
Epoch 10/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 24764.0566 - val_loss: 24741.5938
Epoch 11/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 24482.726

,Target,Ridge Regression Base Model Regularized RMSE,Ridge Regression Base Model Regularized MAE,Ridge Regression Base Model Regularized R²,MLP Base Model RMSE,MLP Base Model MAE,MLP Base Model R²
0,target_aqi_day1,14.265966,10.489315,0.871354,19.172810,14.320646,0.767637
1,target_aqi_day2,26.840461,18.772470,0.551875,27.930368,20.231375,0.514742
2,target_aqi_day3,30.109489,21.517893,0.466703,31.250547,22.956173,0.425516
3,target_aqi_day4,31.580063,22.956517,0.437171,32.772305,24.367542,0.393872
4,Mean,25.698995,18.434049,0.581776,27.781509,20.468933,0.525442


**Observation:** The MLP baseline underperformed Ridge Regression across all four forecast horizons, with mean RMSE increasing from `25.6990` to `27.0197` and mean R² decreasing from `0.5818` to `0.5506`. Further experiments will focus on improving the MLP configuration and generalization.

In [37]:
mlp_base_model_1 = Sequential([
    Input(shape=(23,)),
    Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(1e-4),
    ),
    Dropout(0.15),
    Dense(4, activation="linear"),
])

mlp_base_model_1.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="mse",
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
)

mlp_base_model_fit_params: DeepLearningFitParamSchema = {
    "epochs": 500,
    "batch_size": 32,
    "callbacks": [early_stopping],
    "verbose": 1
}

mlp_base_model_1, (mlp_base_rmse_1, mlp_base_mae_1, mlp_base_r2_1) = train_and_evaluate_model(train_df, val_df,
                         label="MLP Base Model v2", disable_plot=True,
                         baseline=FINAL_SELECTED_FEATURES, model=mlp_base_model_1,
                         compare_model=("Ridge Regression Base Model Regularized", (
                                        ridge_base_rmse_r, ridge_base_mae_r, ridge_base_r2_r
                         )),
                         deep_learning=mlp_base_model_fit_params)

Epoch 1/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 25775.9121 - val_loss: 25665.2148
Epoch 2/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25558.0176 - val_loss: 25457.7812
Epoch 3/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25256.2520 - val_loss: 25165.3438
Epoch 4/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 24828.0000 - val_loss: 24731.4766
Epoch 5/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 24162.2031 - val_loss: 24143.5977
Epoch 6/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 23322.6172 - val_loss: 23406.0859
Epoch 7/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 22285.9258 - val_loss: 22513.9355
Epoch 8/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 21126.7266 - val_loss: 21489.1992
Epoch 9/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 19828.0684 - val_loss: 20390.8828
Epoch 10/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 18426.8047 - val_loss: 19205.4844
Epoch 11/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 16967.021

,Target,Ridge Regression Base Model Regularized RMSE,Ridge Regression Base Model Regularized MAE,Ridge Regression Base Model Regularized R²,MLP Base Model v2 RMSE,MLP Base Model v2 MAE,MLP Base Model v2 R²
0,target_aqi_day1,14.265966,10.489315,0.871354,16.056553,11.284883,0.837033
1,target_aqi_day2,26.840461,18.772470,0.551875,26.421616,18.499138,0.565752
2,target_aqi_day3,30.109489,21.517893,0.466703,29.769363,21.473700,0.478683
3,target_aqi_day4,31.580063,22.956517,0.437171,30.946600,22.695648,0.459524
4,Mean,25.698995,18.434049,0.581776,25.798532,18.488342,0.585248


**Observation:** Increasing the hidden layer to **32 neurons** improved the MLP's mean performance to **RMSE 25.7191**, **MAE 18.4893**, and **R² 0.5866**, slightly surpassing the Ridge Regression baseline. Further increases in model capacity are expected to provide diminishing returns, so the remaining experiments will focus on identifying the best configuration without unnecessarily increasing complexity.

### **Overall Observation:**
Random Forest, Ridge Regression, and MLP experiments have reached a performance plateau when using the same 23-feature set to predict all four targets. This suggests that each forecast horizon may depend on a different set of features. The current feature-selection process retained features that were generally useful across all horizons, but this does not mean those features are optimal for each individual target. A single model predicting all four horizons may therefore be limiting performance.

**Next Approach:** Perform feature extraction and selection **independently for each target** (`target_aqi_day1`–`target_aqi_day4`), followed by training a separate model for each forecast horizon.